In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import re

In [2]:
file_path = "问卷数据400份2.19.xlsx" 

print("正在加载数据...")
try:
    df_raw = pd.read_excel(file_path)
    print(f"读取成功，原始样本量: {len(df_raw)}")
except FileNotFoundError:
    print("❌ 错误：找不到文件，请检查文件名是否正确。")

正在加载数据...
读取成功，原始样本量: 399


In [3]:
# --- 问卷清洗（逻辑：保留0，剔除数值1） ---
trap_cols = [c for c in df_raw.columns if 'Q5' in c and '显示测试' in c]

if not trap_cols:
    print("❌ 错误：未找到陷阱题列，请检查 Excel 表头。")
    valid_df = df_raw.copy()
else:
    trap_col = trap_cols[0]
    total_count = len(df_raw)
    
    # 定义淘汰掩码：数值等于 1 或 1.0 的被淘汰
    eliminated_mask = (df_raw[trap_col] == 1) | (df_raw[trap_col] == 1.0)
    valid_df = df_raw[~eliminated_mask].copy()
    
    # 统计数据
    elim_count = eliminated_mask.sum()
    valid_count = len(valid_df)
    drop_rate = elim_count / total_count

    print(f"{'='*30}")
    print(f"      数据清洗详细报告")
    print(f"{'='*30}")
    print(f"原始样本总数: {total_count}")
    print(f"被淘汰人数 (选B/数值1): {elim_count}")
    print(f"保留样本人数 (包含0, 2, 3): {valid_count}")
    print(f"最终剔除率: {drop_rate:.2%}")
    print(f"{'='*30}")

    # 保存清洗后的原始格式数据
    valid_df.to_csv("cleaned_raw_data.csv", index=False)

      数据清洗详细报告
原始样本总数: 399
被淘汰人数 (选B/数值1): 114
保留样本人数 (包含0, 2, 3): 285
最终剔除率: 28.57%


In [4]:
def parse_attributes_safe(text, is_none=False):
    """
    分段解析属性，防止 LV 级别干扰。
    """
    if is_none:
        return {'Smart': 0, 'Context': 0, 'Privacy': 0, 'Price': 0, 'ASC': 0}
    
    text = str(text).replace('：', ':').replace(' ', '')
    # 按【 】符号切割文本
    segments = re.split(r'【|】', text)
    attrs = {'Smart': 1, 'Context': 1, 'Privacy': 1, 'Price': 0, 'ASC': 1}
    
    for i in range(len(segments)-1):
        key = segments[i]
        val = segments[i+1]
        if '智能' in key:
            if 'LV3' in val or '专家' in val: attrs['Smart'] = 3
            elif 'LV2' in val or '进阶' in val: attrs['Smart'] = 2
        elif '上下文' in key:
            if 'LV3' in val or '超长' in val: attrs['Context'] = 3
            elif 'LV2' in val or '较长' in val: attrs['Context'] = 2
        elif '隐私' in key:
            if 'LV2' in val or '严格保密' in val: attrs['Privacy'] = 2
        elif '价格' in key:
            nums = re.findall(r'(\d+)', val)
            if nums: attrs['Price'] = int(nums[0])
    return attrs

long_data = []
choice_cols = [c for c in df_raw.columns if '方案A' in c and 'Q5' not in c]

for idx, row in valid_df.iterrows():
    respondent_id = row.get('序号', idx)
    for q_col in choice_cols:
        try:
            parts = str(q_col).split('方案B')
            text_a = parts[0].split('方案A')[-1]
            text_b = parts[1]
        except: continue
        
        user_val = row[q_col]
        # 定义三个备选项
        options = [('A', text_a, False), ('B', text_b, False), ('None', '', True)]
        
        for label, text, is_none_flag in options:
            attrs = parse_attributes_safe(text, is_none=is_none_flag)
            is_chosen = 0
            # 映射逻辑：1=B, 2=A, 3=None (根据您的Q5逻辑推导)
            if label == 'B' and user_val == 1: is_chosen = 1
            elif label == 'A' and user_val == 2: is_chosen = 1
            elif label == 'None' and user_val == 3: is_chosen = 1
            
            long_data.append({'ID': respondent_id, 'Q_ID': q_col[:10], 'Choice': is_chosen, **attrs})

df_long = pd.DataFrame(long_data)
df_long.to_csv("choice_long_data.csv", index=False)
print(f"数据重构完成，生成 {len(df_long)} 条观察值。")

数据重构完成，生成 7695 条观察值。


In [5]:
# 准备虚拟变量
df_model = df_long.copy()
for attr in ['Smart', 'Context', 'Privacy']:
    levels = sorted(df_model[attr].unique())
    for lv in levels[1:]: # 以 Level 1 为基准
        if lv > 0:
            df_model[f'{attr}_{lv}'] = (df_model[attr] == lv).astype(int)

# 定义自变量
X_cols = ['ASC', 'Price', 'Smart_2', 'Smart_3', 'Context_2', 'Context_3', 'Privacy_2']
y = df_model['Choice']
X = df_model[X_cols]

print("\n--- 正在运行 Logit 回归 ---")
try:
    result = sm.Logit(y, X).fit(disp=0)
    print(result.summary())
    
    # 计算支付意愿 (WTP)
    beta_price = result.params['Price']
    print(f"\n{'='*30}")
    print(f"      支付意愿 (WTP) 分析报告")
    print(f"{'='*30}")
    
    for col in [c for c in X_cols if c not in ['ASC', 'Price']]:
        wtp = - (result.params[col] / beta_price)
        print(f"[{col}] 对比基准的 WTP: {wtp:.2f} 元/月")
    print(f"{'='*30}")
except Exception as e:
    print(f"模型运行失败: {e}")


--- 正在运行 Logit 回归 ---
                           Logit Regression Results                           
Dep. Variable:                 Choice   No. Observations:                 7695
Model:                          Logit   Df Residuals:                     7688
Method:                           MLE   Df Model:                            6
Date:                Thu, 19 Feb 2026   Pseudo R-squ.:                -0.05247
Time:                        22:39:55   Log-Likelihood:                -5141.7
converged:                       True   LL-Null:                       -4885.4
Covariance Type:            nonrobust   LLR p-value:                     1.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ASC           -0.5918      0.097     -6.074      0.000      -0.783      -0.401
Price          0.0013      0.001      1.535      0.125      -0.000       0.003
Smart_2       -0.1035      0.